# SFT Distillation: VibeThinker-3B on FABLE-5 (Complete 2M traces) — Unsloth

**Author:** Behrooz Azarkhalili

| Feature | Detail |
|---------|--------|
| Base | `WeiboAI/VibeThinker-3B` |
| Dataset | `ermiaazarkhalili/Fable-5-Complete-2M-Clean` |
| Method | LoRA (QLoRA 4-bit), assistant-only masking |
| Rank / α | r=16, α=16 |
| LR / seq | 0.0002, max_seq 4096 |
| Recipe | docs/recipes/fable-mythos-distillation.md §3.1 (Qwable-9B-aligned) |


In [ ]:
# ============================================================================
# Install (uncomment for Colab / bare-metal)
# ============================================================================
# !pip install unsloth datasets tqdm

# SLURM (Fir cluster):
#   module load gcc arrow python/3.11.5
#   source /scratch/$USER/venvs/hf_unsloth/bin/activate

In [ ]:
from __future__ import annotations
import json, os, torch

SMOKE_TEST: bool = False  # Set False for full training

MODEL_NAME = "WeiboAI/VibeThinker-3B"
# MODEL_NAME = "WeiboAI/VibeThinker-3B"  # Alternative
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True
HUB_MODEL_ID = "ermiaazarkhalili/VibeThinker-3B-SFT-Fable5"
LORA_R = 16
LORA_ALPHA = 16
LEARNING_RATE = 0.0002
MAX_STEPS = 5 if SMOKE_TEST else -1
NUM_EPOCHS = 1 if SMOKE_TEST else 2
BATCH_SIZE = 2
GRAD_ACCUM = 4


# ── Checkpoint + structured logging (absolute paths; F3 avoidance) ─────
SLUG = "fable-vibethinker-3b"
JOB_ID = os.environ.get("SLURM_JOB_ID", "local")
CHECKPOINT_ROOT = os.environ.get(
    "CHECKPOINT_ROOT",
    f"/scratch/{os.environ.get('USER','ermia')}/checkpoints/{SLUG}-{JOB_ID}",
)
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)
TB_DIR = f"{CHECKPOINT_ROOT}/tb"
JSONL_LOG = f"{CHECKPOINT_ROOT}/train_log.jsonl"
SAVE_STEPS = (150 if not SMOKE_TEST else 999_999)
print(f"[OK] Checkpoints → {CHECKPOINT_ROOT}")

print(f"[OK] Model: {MODEL_NAME}")
print(f"[OK] SMOKE_TEST: {SMOKE_TEST}, max_steps: {MAX_STEPS}")
print(f"[OK] PyTorch: {torch.__version__}")
print(f"[OK] GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# ============================================================================
# HF Authentication
# ============================================================================
try:
    from dotenv import load_dotenv; load_dotenv()
except ImportError: pass
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception: pass
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("[OK] Authenticated with HF Hub")
else:
    print("[WARN] No HF_TOKEN found")

In [ ]:
# ============================================================================
# Load Model with Unsloth
# ============================================================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)
print(f"[OK] Model loaded: {MODEL_NAME}")

# Granite 4.1 returns a plain tokenizer (probe-verified GPT2Tokenizer) — no
# VL-processor unwrap needed.

In [ ]:
# ============================================================================
# Apply LoRA
# ============================================================================
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none", random_state=3407,
)
print(f"[OK] LoRA applied: r={LORA_R}, alpha={LORA_ALPHA}")

In [ ]:
# ============================================================================
# Chat Template (VibeThinker-3B ships a built-in chat_template in the tokenizer)
# ============================================================================
print(f"[OK] Using built-in VibeThinker-3B chat template")

In [ ]:
# ============================================================================
# Load & Process FABLE Trace Dataset (D) — completion-path
# ============================================================================
# The clean fable datasets store `messages` as a JSON STRING, and tool_use
# assistant turns have EMPTY content. The trainable signal is the `completion`
# field (= <think>..</think> + response, tool-calls inline). We rebuild a clean
# 2-turn chat [user=context, assistant=completion] and render it through THIS
# model's own chat template so assistant-turn markers exist for response-masking.
from datasets import load_dataset
import json as _json, hashlib as _hashlib

dataset = load_dataset("ermiaazarkhalili/Fable-5-Complete-2M-Clean", split="train")
print(f"[OK] Loaded {len(dataset):,} fable samples from ermiaazarkhalili/Fable-5-Complete-2M-Clean")

# ── Held-out eval split (Qwable-style) — EXCLUDE from training ───────────────
# Deterministic ~5pct holdout keyed on a stable per-row id, so the eval script can
# reconstruct the SAME complement without a separately-pushed dataset. Key =
# session_id if present, else a hash of the completion (D rows). hash mod 20 == 0.
def _holdout_key(sample):
    sid = str(sample.get("session_id") or "")
    return sid if sid else "C:" + str(sample.get("completion") or "")[:200]

def _is_holdout(sample):
    return int(_hashlib.md5(_holdout_key(sample).encode()).hexdigest(), 16) % 20 == 0

_n_all = len(dataset)
dataset = dataset.filter(lambda s: not _is_holdout(s))
_pct = 100 * (_n_all - len(dataset)) / max(_n_all, 1)
print(f"[OK] Excluded held-out eval split: train={len(dataset):,} / "
      f"held-out={_n_all - len(dataset):,} ({_pct:.1f} pct)")

if SMOKE_TEST:
    dataset = dataset.select(range(min(100, len(dataset))))
    print(f"[SMOKE] Truncated to {len(dataset)} samples")

def _first_user(sample):
    """Extract the user prompt: prefer messages[] (JSON string), fall back to context."""
    raw = sample.get("messages")
    if isinstance(raw, str) and raw.strip():
        try:
            for m in _json.loads(raw):
                if isinstance(m, dict) and m.get("role") == "user" and m.get("content"):
                    return str(m["content"])
        except Exception:
            pass
    return str(sample.get("context") or "")

def format_fable(sample):
    user = _first_user(sample).strip()
    completion = str(sample.get("completion") or "").strip()
    msgs = [
        {"role": "user", "content": user},
        {"role": "assistant", "content": completion},
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {"text": text}

def _has_target(sample):
    # Filter on the COMPLETION (the trainable assistant target), NOT the rendered
    # text length — a long user context must not let an EMPTY completion survive
    # (xmodel-review finding #5). Require a non-trivial completion.
    return len(str(sample.get("completion") or "").strip()) > 16

_n_before = len(dataset)
dataset = dataset.filter(_has_target)
dataset = dataset.map(format_fable)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
print(f"[OK] Processed: {len(dataset):,} samples (dropped {_n_before - len(dataset):,} empty-target)")
assert len(dataset) > 0, "Dataset is EMPTY after filtering — no trainable rows; refusing to train."
print("[sample] first 300 chars:\n", dataset[0]["text"][:300])


In [ ]:
import os as _os
# ============================================================================
# Training
# ============================================================================
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset, eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5, max_steps=MAX_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        optim="adamw_8bit", weight_decay=0.001,
        lr_scheduler_type="linear", seed=3407,
        # ── Absolute paths (F3 avoidance) ──
        output_dir=CHECKPOINT_ROOT,
        logging_dir=TB_DIR,
        # ── Checkpoints: periodic snapshots, rolling keep ──
        save_strategy=("no" if SMOKE_TEST else "steps"),
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        # ── Streaming logs: every step, first step included ──
        logging_strategy="steps",
        logging_steps=1,
        logging_first_step=True,
        logging_nan_inf_filter=True,
        report_to="none",
    ),
)

# ── Per-step JSONL log persistence (survives kill/timeout) ─────────────
import json as _json, time as _time
from transformers import TrainerCallback

class JsonlLoggerCallback(TrainerCallback):
    """Append every logged metric dict as a JSON line to `path`."""
    def __init__(self, path: str):
        self.path = path
        # Touch file so absence-of-file vs zero-steps is unambiguous.
        open(self.path, "a").close()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        rec = {"ts": _time.time(), "step": state.global_step, **logs}
        with open(self.path, "a") as f:
            f.write(_json.dumps(rec) + "\n")

trainer.add_callback(JsonlLoggerCallback(JSONL_LOG))
print(f"[OK] JSONL log: {JSONL_LOG}")

gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
start_mem = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"[OK] GPU: {gpu_stats.name}, {max_memory} GB, reserved: {start_mem} GB")
print(f"[....] Training (max_steps={MAX_STEPS}, num_epochs={NUM_EPOCHS}) ...")


# ── Assistant-only loss masking (train_on_responses_only) — fable trace-SFT ──
# Mask prompt + tool-result tokens so loss is computed ONLY on assistant turns
# (Qwable-9B / recipe §3.3). Markers derived at runtime by diffing a chat
# rendered with vs without the generation prompt — correct for every family.
from unsloth.chat_templates import train_on_responses_only

def _derive_parts(tok):
    # IMPORTANT: derive markers from the SAME object the dataset cell renders with
    # (the module-level `tokenizer`), NOT a re-unwrapped inner — otherwise Gemma
    # (whose `tokenizer` is the get_chat_template-wrapped processor) would diff a
    # different template than the data was rendered with → all-masked → abort.
    sentinel = "\u0000USR\u0000"
    base = [{"role": "user", "content": sentinel}]
    no_gen = tok.apply_chat_template(base, tokenize=False, add_generation_prompt=False)
    with_gen = tok.apply_chat_template(base, tokenize=False, add_generation_prompt=True)
    # Guard (xmodel-review finding #3): if the template stripped/escaped the NUL
    # sentinel, split() yields the WHOLE prompt as inst — a marker that won't match
    # real rows. Detect by requiring the sentinel to actually appear in no_gen.
    if sentinel not in no_gen:
        raise RuntimeError(
            "chat template did not preserve the NUL sentinel — cannot derive masking "
            "markers reliably. Inspect the rendered template for this model."
        )
    resp = with_gen[len(no_gen):] if with_gen.startswith(no_gen) else with_gen.split(sentinel)[-1]
    inst = no_gen.split(sentinel)[0]
    resp = resp.strip("\n")
    # Thinking-model templates (e.g. Qwen3.5-0.8B/2B) auto-inject an EMPTY
    # "<think>\n\n</think>" scaffold right after the assistant role marker when
    # add_generation_prompt=True. Our real rows carry REAL <think> content, so a
    # response_part containing the empty scaffold matches NOTHING → everything is
    # masked → 0 trainable rows → "num_samples=0". Key masking on the assistant
    # ROLE marker only: truncate response_part at the first "<think>".
    _ti = resp.find("<think>")
    if _ti != -1:
        resp = resp[:_ti].rstrip("\n")
    return inst.strip("\n"), resp

_inst_part, _resp_part = _derive_parts(tokenizer)
print(f"[mask] instruction_part={_inst_part!r}")
print(f"[mask] response_part={_resp_part!r}")
assert _resp_part, "empty response_part — cannot mask"

trainer = train_on_responses_only(
    trainer, instruction_part=_inst_part, response_part=_resp_part,
)

# Sanity gate: run the collator on a couple of examples and count unmasked
# (!= -100) label tokens. All-masked => marker mismatch => silent zero signal.
# This is BEST-EFFORT verification: the collator's exact input shape varies by
# TRL/unsloth version, so any failure to introspect is a WARNING (never a crash)
# — a false gate-crash would block a perfectly good run. The real backstop is
# train loss being finite and > 0, which the JSONL logger captures per step.
import numpy as _np
try:
    _rows = [trainer.train_dataset[i] for i in range(min(2, len(trainer.train_dataset)))]
    _batch = trainer.data_collator(_rows)
    _lab = _batch["labels"]
    _labels = _lab.cpu().numpy() if hasattr(_lab, "cpu") else _np.asarray(_lab)
    _kept = int((_labels != -100).sum())
    print(f"[mask] unmasked label tokens across {len(_rows)} samples: {_kept}")
    if _kept == 0:
        print("[mask][WARN] ALL labels masked — response_part marker may not match the "
              "rendered template. Training would learn nothing. Inspect _resp_part above.")
    else:
        print("[OK] Assistant-only masking applied + verified (>0 unmasked tokens)")
except Exception as _e:
    print(f"[mask][WARN] could not introspect collator labels ({type(_e).__name__}: {_e}); "
          "masking IS applied, verification skipped. Watch first-step train_loss > 0.")

trainer_stats = trainer.train(resume_from_checkpoint=(_os.path.isdir(CHECKPOINT_ROOT) and any(d.startswith('checkpoint-') for d in _os.listdir(CHECKPOINT_ROOT))))

used = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n[OK] Training complete!")
print(f"  Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"  Loss: {trainer_stats.metrics.get('train_loss', 'N/A')}")
print(f"  Peak VRAM: {used} GB ({round(used/max_memory*100, 1)}%)")

In [ ]:
# ============================================================================
# Inference Test
# ============================================================================
test_prompts = [
    "Check if the numbers 8 and 1233 are powers of two.",
    "What's the weather like in New York today?",
    "Calculate the average of: 10, 20, 30, 40, 50",
]

print("[....] Testing ...\n")
for i, prompt in enumerate(test_prompts, 1):
    print(f"{'=' * 50}")
    print(f"Test {i}: {prompt}")
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = model.generate(
        **tokenizer(text, return_tensors="pt").to("cuda"),
        max_new_tokens=4096, temperature=0.7, top_p=0.8, top_k=20, use_cache=True,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the generated part
    if prompt in response:
        response = response.split(prompt)[-1]
    print(response[:300])
    print()

In [ ]:
import os as _os
_SCRATCH = _os.environ.get("SCRATCH", "/scratch/" + _os.environ.get("USER", "ermia"))
_FABLE_OUT = _os.path.join(_SCRATCH, "fable-artifacts", "vibethinker_3b_d")
_os.makedirs(_os.path.dirname(_FABLE_OUT), exist_ok=True)
# ============================================================================
# Save & Export
# ============================================================================
model.save_pretrained((_FABLE_OUT + "_lora"))
tokenizer.save_pretrained((_FABLE_OUT + "_lora"))
print("[OK] LoRA adapter saved")

if not SMOKE_TEST:
    model.save_pretrained_merged((_FABLE_OUT + "_merged"), tokenizer)
    print("[OK] Merged model saved")
    _pushed_merged = False
    model.push_to_hub_merged(HUB_MODEL_ID, tokenizer, token=os.environ.get("HF_TOKEN", ""))
    print(f"[OK] Pushed to {HUB_MODEL_ID}")
    _pushed_merged = True
    _pushed_gguf = False
    try:
        model.save_pretrained_gguf((_FABLE_OUT + "_gguf"), tokenizer, quantization_method=["q4_k_m", "q5_k_m", "q8_0"])
        print("[OK] GGUF saved")
        model.push_to_hub_gguf(f"{HUB_MODEL_ID}-GGUF", tokenizer, quantization_method=["q4_k_m", "q5_k_m", "q8_0"], token=os.environ.get("HF_TOKEN", ""))
        print(f"[OK] GGUF pushed")
        _pushed_gguf = True
    except Exception as e:
        print(f"[WARN] GGUF export failed (non-fatal): {e}")
        print("[INFO] Training + merge + Hub push succeeded. GGUF can be done separately.")
else:
    print("[SMOKE] Skipping merge/GGUF/push")

print("\n[OK] Done!")

In [ ]:
# ============================================================================
# Hub Round-Trip Validation (non-smoke only)
# ============================================================================
# Loads the pushed merged repo via transformers AutoModel, runs a small
# generation test, then downloads one quantized GGUF and runs it through
# llama-cli to prove the published artifacts are usable by downstream consumers.
if not SMOKE_TEST and globals().get("_pushed_merged", False):
    import gc, shutil, subprocess, tempfile, traceback
    from pathlib import Path as _Path

    _VAL_PROMPT = "The capital of France is"
    _VAL_MAX_TOKENS = 4096
    _LLAMA_CLI = "/home/ermia/.unsloth/llama.cpp/build/bin/llama-cli"

    # Free GPU memory held by the training session before reload
    try:
        del model; del trainer  # noqa: F821
    except Exception:
        pass
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

    print(f"\n[VALIDATE] Round-trip test for {HUB_MODEL_ID}")

    # --- Merged safetensors round-trip ---
    merged_ok = False
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        _tok = AutoTokenizer.from_pretrained(HUB_MODEL_ID, trust_remote_code=True)
        _mdl = AutoModelForCausalLM.from_pretrained(
            HUB_MODEL_ID,
            torch_dtype="auto",
            device_map="auto",
            trust_remote_code=True,
        )
        _inputs = _tok(text=_VAL_PROMPT, return_tensors="pt").to(_mdl.device)
        _out = _mdl.generate(**_inputs, max_new_tokens=_VAL_MAX_TOKENS, do_sample=False)
        _decoded = _tok.decode(_out[0], skip_special_tokens=True)
        if len(_decoded.strip()) > len(_VAL_PROMPT):
            merged_ok = True
            print(f"[VALIDATE] Merged repo inference OK")
            print(f"           prompt: {_VAL_PROMPT!r}")
            print(f"           output: {_decoded!r}")
        else:
            print("[VALIDATE] Merged inference returned empty or prompt-only output")
        del _mdl, _tok
        gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception:
            pass
    except Exception as _e:
        print(f"[VALIDATE] Merged repo load/inference FAILED: {_e}")
        traceback.print_exc()

    # --- GGUF round-trip (only if GGUF push succeeded) ---
    gguf_ok = False
    if globals().get("_pushed_gguf", False):
        try:
            from huggingface_hub import hf_hub_download, list_repo_files
            _gguf_repo = f"{HUB_MODEL_ID}-GGUF"
            _files = list_repo_files(_gguf_repo)
            # Prefer q4_k_m for speed; fall back to first .gguf
            _ggufs = [f for f in _files if f.lower().endswith(".gguf") and "mmproj" not in f.lower()]
            _chosen = next((f for f in _ggufs if "q4_k_m" in f.lower()), _ggufs[0] if _ggufs else None)
            if _chosen is None:
                print(f"[VALIDATE] No .gguf files found on {_gguf_repo}")
            elif not _Path(_LLAMA_CLI).exists():
                print(f"[VALIDATE] llama-cli not found at {_LLAMA_CLI} — skipping GGUF inference")
            else:
                print(f"[VALIDATE] Downloading {_chosen} from {_gguf_repo} ...")
                _local = hf_hub_download(_gguf_repo, _chosen, token=os.environ.get("HF_TOKEN", None))
                print(f"[VALIDATE] Running llama-cli on {_Path(_local).name} ...")
                _cmd = [
                    _LLAMA_CLI, "-m", _local,
                    "-p", _VAL_PROMPT,
                    "-n", str(_VAL_MAX_TOKENS),
                    "--no-display-prompt",
                    "--no-warmup",
                    "-no-cnv",
                ]
                _r = subprocess.run(_cmd, capture_output=True, text=True, timeout=300)
                if _r.returncode == 0 and len(_r.stdout.strip()) > 0:
                    gguf_ok = True
                    print(f"[VALIDATE] GGUF inference OK")
                    print(f"           llama-cli stdout (first 200c): {_r.stdout[:200]!r}")
                else:
                    print(f"[VALIDATE] GGUF inference FAILED rc={_r.returncode}")
                    print(f"           stderr (last 300c): {_r.stderr[-300:]!r}")
        except Exception as _e:
            print(f"[VALIDATE] GGUF round-trip FAILED: {_e}")
            traceback.print_exc()

    print()
    print("=" * 60)
    print(f"[VALIDATE] SUMMARY for {HUB_MODEL_ID}")
    print(f"           merged repo : {'PASS' if merged_ok else 'FAIL'}")
    print(f"           GGUF repo   : {'PASS' if gguf_ok else ('FAIL' if globals().get('_pushed_gguf', False) else 'SKIP (no GGUF pushed)')}")
    print("=" * 60)
elif SMOKE_TEST:
    print("[SMOKE] Skipping Hub round-trip validation")
else:
    print("[VALIDATE] Skipped — no Hub push occurred (training or merge likely failed)")
